### Created on 10/20/2025 by MTH 
# Notebook to use to generate csv input files for the MCMC model 
The idea with this notebook is that is should be a clear way to generate and save new CSV files to read into the MCMC model without just copying and pasting some messy CSV files and notes. 


In [1]:
# Import pytorch libraries 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import sys
import os
import glob
from tqdm import tqdm 

from scipy.optimize import curve_fit


# Add the directory containing the module to sys.path
module_path = os.path.abspath("/crucial/modified_MCMC/dATP_multiscale_modeling/resultsOrg")
if module_path not in sys.path:
    sys.path.append(module_path)

# Import the module
import helper_functions as hf

In [2]:
previous_csv = pd.read_csv('/crucial/modified_MCMC/dATP_multiscale_modeling/MCMC_simulation_results/2025-10-09_1436/k2_k2_k4_plus_drug_testing.csv', comment = '#', 
    nrows = 1)

previous_csv


,protocol,k_force_baseline,k_force_drug,k_plus_SR_baseline,k_plus_SR_drug,k_minus_SR,k_xb,k1_plus_ref_baseline,k1_plus_ref_drug,k2_plus_baseline,...,conc_ATP,conc_Pi,delta_G_ATP,alpha,beta,eta,g_Cb,g_Ca,k_plus_SS,k_minus_SS
0,1,0.2,0.2,16,16,15,5,0.001421,0.001421,0.026875,...,0.005,0.003,-13,0.28,0.35,0.68,0,1,0,0


In [3]:

# Make sure that all the druc rates match between baseline and drug 
for i in range(1,len(previous_csv.columns)):
    if 'drug' in previous_csv.columns[i]:
        if 'baseline' in previous_csv.columns[i -1]:
            print("Checkout out for column ", previous_csv.columns[i], " and ", previous_csv.columns[i-1])
            if previous_csv.iloc[0,i] != previous_csv.iloc[0,i-1]:
                print("Values don't match! ", previous_csv.iloc[0,i], " and ", previous_csv.iloc[0,i-1])
                print("Setting values to match now: ")
                previous_csv.iloc[0,i] = previous_csv.iloc[0,i-1]
                print("New values: ", previous_csv.iloc[0,i], " and ", previous_csv.iloc[0,i-1])
        else: 
            print("Didn't checkout for column ", previous_csv.columns[i], " and ", previous_csv.columns[i-1])

Checkout out for column  k_force_drug  and  k_force_baseline
Checkout out for column  k_plus_SR_drug  and  k_plus_SR_baseline
Checkout out for column  k1_plus_ref_drug  and  k1_plus_ref_baseline
Checkout out for column  k2_plus_drug  and  k2_plus_baseline
Values don't match!  0.05  and  0.026874763
Setting values to match now: 
New values:  0.026874763  and  0.026874763
Checkout out for column  k3_plus_drug  and  k3_plus_baseline
Checkout out for column  k4_plus_ref_drug  and  k4_plus_ref_baseline
Didn't checkout for column  percent_drug  and  kCa_minus_ref


In [4]:
print(previous_csv.to_csv())

,protocol,k_force_baseline,k_force_drug,k_plus_SR_baseline,k_plus_SR_drug,k_minus_SR,k_xb,k1_plus_ref_baseline,k1_plus_ref_drug,k2_plus_baseline,k2_plus_drug,k3_plus_baseline,k3_plus_drug,k4_plus_ref_baseline,k4_plus_ref_drug,kB_plus_ref,kB_minus_ref,kCa_plus_ref,kCa_minus_ref,percent_drug,lambda,gamma_B,gamma_M,mu_B,mu_M,q,r,x_preR,x_xb,conc_ADP,conc_ATP,conc_Pi,delta_G_ATP,alpha,beta,eta,g_Cb,g_Ca,k_plus_SS,k_minus_SS
0,1,0.2,0.2,16,16,15,5,0.00142101,0.00142101,0.026874763,0.026874763,0.00997169,0.00997169,0.111710515,0.111710515,8.900016461,0.1,0.09,0.778868408,0,0,21.22374456,21,21,5.402302855,1,1,0,0.075,3e-05,0.005,0.003,-13,0.28,0.35,0.68,0,1,0,0



In [5]:
# Argument dictionary witll be set as some 


def add_parameter_experiments(df, experiments):
    """
    Add new parameter experiments to the dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The existing dataframe with MCMC parameters
    experiments : dict or list of dicts
        Single dict or list of dicts containing parameter values to modify.
        Only include parameters you want to change from the baseline (row 0).
        
    Returns:
    --------
    pd.DataFrame
        Updated dataframe with new experiments appended
        
    Examples:
    ---------
    # Single experiment, changing one parameter
    df = add_parameter_experiments(df, {'k_force_baseline': 0.3})
    
    # Single experiment, changing multiple parameters
    df = add_parameter_experiments(df, {
        'k_force_baseline': 0.3,
        'k_force_drug': 0.25,
        'percent_drug': 0.5
    })
    
    # Multiple experiments at once
    df = add_parameter_experiments(df, [
        {'k_force_baseline': 0.3},
        {'k_force_baseline': 0.4},
        {'k_force_baseline': 0.5, 'percent_drug': 0.3}
    ])
    """
    # Convert single dict to list for uniform processing
    if isinstance(experiments, dict):
        experiments = [experiments]
    
    # Get baseline parameters (assumes row 0 is baseline)
    baseline = df.iloc[0].to_dict()
    
    # Create new rows
    new_rows = []
    for exp in experiments:
        # Start with baseline values
        new_row = baseline.copy()
        # Update with new parameter values
        new_row.update(exp)
        new_rows.append(new_row)
    
    # Create new dataframe with new rows
    new_df = pd.DataFrame(new_rows)
    
    # Append to original dataframe and reset index
    result_df = pd.concat([df, new_df], ignore_index=True)
    
    return result_df


# Alternative: Generate grid of parameter combinations
def add_parameter_grid(df, param_grid):
    """
    Add experiments for all combinations of parameter values (grid search).
    
    Parameters:
    -----------
    df : pd.DataFrame
        The existing dataframe with MCMC parameters
    param_grid : dict
        Dictionary mapping parameter names to lists of values to try
        
    Returns:
    --------
    pd.DataFrame
        Updated dataframe with all parameter combinations appended
        
    Example:
    --------
    df = add_parameter_grid(df, {
        'k_force_baseline': [0.2, 0.3, 0.4],
        'percent_drug': [0.0, 0.5, 1.0]
    })
    # This creates 3 x 3 = 9 new experiments
    """
    import itertools
    
    # Get all combinations
    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())
    combinations = list(itertools.product(*param_values))
    
    # Create experiment dicts
    experiments = [dict(zip(param_names, combo)) for combo in combinations]
    
    return add_parameter_experiments(df, experiments)



In [10]:
drug_conc = add_parameter_grid(previous_csv, {
    'percent_drug': [0.3, 1, 3, 30],
})

drug_conc.to_csv('temp_output/high_conc_afi_test.csv', index=False)
